In [ ]:
'''
Santa 2026 | Shared Helper Search

Search for a different intermediate value that can serve both
consumers of an existing helper. Match inverse arithmetic routes, remove
unused ancestors and keep the original order of the surviving operations.
One branch may use an extra step when both branches share an existing value.

The default pilot searches the same 60 puzzles in both notebooks.
A shorter portfolio is exported with all 1,000 scored puzzles.
Set PUZZLE_LIMIT to None to search the pinned 128,485-instruction portfolio.
'''

In [ ]:
'''
Credits

Ertuğrul Özer (ozertuu) supplied the public portfolio used as input.
All unmodified programs in the output are from his dataset, version 506.
The dataset listing reports its license as Unknown.
https://www.kaggle.com/datasets/ozertuu/santa-2026-verified-programs/versions/506

The baseline loading and replay build on our adaptation of his Apache 2.0
Starter Notebook.
https://www.kaggle.com/code/ozertuu/santa-2026-starter-notebook

Asterios Panoras published the shared-helper experiments that informed the
design of this search. Alex Chilton's Rudolph notebook informed the register
allocation experiment. The search and scheduling code here are our own.
https://www.kaggle.com/datasets/asteriospanoras/santa-2026-elf-programs-and-experiments
https://www.kaggle.com/code/alexchilton/rudolph-the-red-nosed-register-allocator
'''

## 1 · Configuration

In [ ]:
'''
Local runs look for the v506 portfolio in santa-2026/data.
Kaggle runs use the two attachments declared in the notebook metadata.
'''

EXPERIMENT_SLUG = '03_shared_helper'
ALLOW_REORDER = False
PUZZLE_LIMIT = 60
HELPER_LIMIT = 28
HELPER_VALUE_LIMIT = 32
ROUTES_PER_VALUE = 2
EXTENDED_ROUTES_PER_ANCHOR = 4

In [ ]:
import csv
import hashlib
import json
from collections import Counter
from dataclasses import dataclass
from math import isqrt
from pathlib import Path
from statistics import median
from tempfile import TemporaryDirectory
from time import perf_counter


COMPETITION = 'santa-2026'
DATASET_OWNER = 'ozertuu'
DATASET_SLUG = 'santa-2026-verified-programs'
DATASET_VERSION = 506
PORTFOLIO_SHA256 = '4560d52245b3eca32f229a03a26b368c1fb506e0aa3ffa950607434e5b9bd0e7'
EXPECTED_SCORE = 128_485
SUBMISSION_COLUMNS = ['id', 'program']
MAX_INSTRUCTIONS = 4_096
MAX_PROGRAM_BYTES = 131_072

Instruction = tuple[str, int, int, int]
Program = tuple[Instruction, ...]


@dataclass(frozen=True)
class Config:
    puzzles_csv: Path
    portfolio_csv: Path
    output_csv: Path

    @classmethod
    def discover(cls) -> 'Config':
        kaggle = Path('/kaggle/input')
        if kaggle.is_dir():
            puzzles = cls.locate(kaggle, COMPETITION, 'puzzles.csv')
            programs = cls.locate(kaggle, DATASET_SLUG, 'best_programs.csv')
            return cls(puzzles, programs, Path('/kaggle/working/submission.csv'))

        root = Path.cwd()
        competition_dir = root / 'competitions' / 'public-comp' / COMPETITION
        if competition_dir.is_dir():
            root = competition_dir
        elif root.name == EXPERIMENT_SLUG and root.parent.name == COMPETITION:
            root = root.parent
        data = root / 'data'
        return cls(
            data / 'puzzles.csv',
            data / DATASET_SLUG / 'best_programs.csv',
            root / 'working' / EXPERIMENT_SLUG / 'submission.csv',
        )

    @staticmethod
    def locate(root: Path, slug: str, filename: str) -> Path:
        matches = [path for path in root.rglob(filename) if slug in path.parent.parts]
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Expected one {slug}/{filename} input, found {len(matches)}'
            )
        return matches[0]

    def check_inputs(self) -> None:
        if not self.puzzles_csv.is_file():
            raise FileNotFoundError(
                f'Attach the {COMPETITION} competition data or copy puzzles.csv '
                f'to {self.puzzles_csv}'
            )
        if not self.portfolio_csv.is_file():
            raise FileNotFoundError(
                f'Attach {DATASET_OWNER}/{DATASET_SLUG} version {DATASET_VERSION} '
                f'or copy best_programs.csv to {self.portfolio_csv}'
            )
        digest = hashlib.sha256(self.portfolio_csv.read_bytes()).hexdigest()
        if digest != PORTFOLIO_SHA256:
            raise ValueError(
                f'Expected {DATASET_SLUG} version {DATASET_VERSION}. '
                f'Unexpected SHA-256 for {self.portfolio_csv}: {digest}'
            )

## 2 · Exact replay and CSV I/O

In [ ]:
@dataclass(frozen=True)
class Puzzle:
    starts: tuple[int, ...]
    targets: frozenset[int]
    value_limit: int
    register_count: int

    @classmethod
    def from_row(cls, row: dict[str, str]) -> 'Puzzle':
        return cls(
            tuple(json.loads(row['starting_values'])),
            frozenset(json.loads(row['targets'])),
            int(row['value_limit']),
            int(row['registers']),
        )


@dataclass(frozen=True)
class ProgramProfile:
    length: int
    operations: Counter[str]
    peak_value: int
    populated_registers: int


class Calculator:
    '''Read both sources before writing and remember targets across overwrites.'''

    def replay(self, puzzle: Puzzle, program: Program) -> ProgramProfile:
        if len(program) > MAX_INSTRUCTIONS:
            raise ValueError('Program exceeds the instruction limit')

        registers = list(puzzle.starts) + [None] * (
            puzzle.register_count - len(puzzle.starts)
        )
        missing = set(puzzle.targets).difference(puzzle.starts)
        operations = Counter()
        peak_value = max(puzzle.starts, default=0)

        for number, instruction in enumerate(program, start=1):
            if len(instruction) != 4:
                raise ValueError(f'Step {number}: expected four instruction fields')
            op, left, right, dest = instruction
            if any(type(index) is not int or not 0 <= index < puzzle.register_count
                   for index in (left, right, dest)):
                raise ValueError(f'Step {number}: register index out of range')

            a, b = registers[left], registers[right]
            if a is None or b is None:
                raise ValueError(f'Step {number}: undefined source register')
            if op == '+':
                value = a + b
            elif op == '-':
                value = a - b
            elif op == '*':
                value = a * b
            elif op == '/' and a % b == 0:
                value = a // b
            else:
                raise ValueError(f'Step {number}: unsupported operation or inexact division')

            if not 1 <= value <= puzzle.value_limit:
                raise ValueError(f'Step {number}: result outside the value limit')
            registers[dest] = value
            missing.discard(value)
            operations[op] += 1
            peak_value = max(peak_value, value)

        if missing:
            raise ValueError(f'{len(missing)} missing targets')

        '''
        Defined registers stay populated after a write, so the final count
        is also the peak populated count for the program.
        '''
        return ProgramProfile(
            len(program), operations, peak_value,
            sum(value is not None for value in registers),
        )

In [ ]:
class PortfolioPipeline:
    def __init__(self, config: Config):
        self.config = config
        self.calculator = Calculator()

    def load_puzzles(self) -> dict[str, Puzzle]:
        with self.config.puzzles_csv.open(encoding='utf-8-sig', newline='') as file:
            rows = list(csv.DictReader(file))
        puzzles = {row['id']: Puzzle.from_row(row) for row in rows}
        expected = {f'scored_{index:04d}' for index in range(1, 1_001)}
        if len(rows) != len(expected) or puzzles.keys() != expected:
            raise ValueError('Expected the 1,000 scored puzzle IDs exactly once')
        return puzzles

    def verify(
        self, path: Path, puzzles: dict[str, Puzzle],
    ) -> tuple[dict[str, Program], dict[str, ProgramProfile]]:
        programs, profiles = {}, {}
        with path.open(encoding='utf-8', newline='') as file:
            reader = csv.reader(file)
            if next(reader, None) != SUBMISSION_COLUMNS:
                raise ValueError(f'{path}: expected the id,program header')
            for row in reader:
                if len(row) != 2:
                    raise ValueError(f'{path}: expected two fields at line {reader.line_num}')
                puzzle_id, cell = row
                if puzzle_id not in puzzles or puzzle_id in programs:
                    raise ValueError(f'{path}: unexpected or duplicate ID {puzzle_id}')
                if len(cell.encode('utf-8')) > MAX_PROGRAM_BYTES:
                    raise ValueError(f'{puzzle_id}: program cell exceeds the byte limit')
                try:
                    payload = json.loads(cell)
                    if type(payload) is not list or any(type(step) is not list for step in payload):
                        raise ValueError('Expected a JSON array of instruction arrays')
                    program = tuple(tuple(step) for step in payload)
                    profiles[puzzle_id] = self.calculator.replay(puzzles[puzzle_id], program)
                except ValueError as error:
                    raise ValueError(f'{puzzle_id}: {error}') from error
                programs[puzzle_id] = program
        if programs.keys() != puzzles.keys():
            raise ValueError(f'{path}: portfolio is missing scored puzzle IDs')
        return programs, profiles

    def export(self, programs: dict[str, Program], puzzles: dict[str, Puzzle]) -> None:
        output = self.config.output_csv
        output.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(dir=output.parent) as directory:
            temporary = Path(directory) / output.name
            with temporary.open('w', encoding='utf-8', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(SUBMISSION_COLUMNS)
                for puzzle_id in sorted(programs):
                    cell = json.dumps(programs[puzzle_id], separators=(',', ':'))
                    writer.writerow((puzzle_id, cell))
            written, _ = self.verify(temporary, puzzles)
            if written != programs:
                raise RuntimeError('The exported programs differ from the verified input')
            temporary.replace(output)

## 3 · Arithmetic graph and search

In [ ]:
@dataclass(frozen=True)
class Operation:
    operator: str
    left: int
    right: int


@dataclass
class ProgramGraph:
    values: dict[int, int]
    recipes: dict[int, Operation]
    roots: set[int]
    users: dict[int, set[int]]

    @classmethod
    def from_program(cls, puzzle: Puzzle, program: Program) -> 'ProgramGraph':
        registers = [-index - 1 for index in range(len(puzzle.starts))]
        registers += [None] * (puzzle.register_count - len(registers))
        values = {-index - 1: value for index, value in enumerate(puzzle.starts)}
        recipes = {}
        users = {node: set() for node in values}
        seen = set(puzzle.starts)
        roots = set()

        '''
        Node IDs leave room to insert one helper immediately before a consumer.
        Each node identifies a single write, even when a register is overwritten.
        '''
        for index, (operator, a, b, dest) in enumerate(program):
            node = index * 2 + 1
            left, right = registers[a], registers[b]
            value = SharedHelperSearch.apply(operator, values[left], values[right])
            values[node] = value
            recipes[node] = Operation(operator, left, right)
            users[node] = set()
            users[left].add(node)
            users[right].add(node)
            if value in puzzle.targets and value not in seen:
                roots.add(node)
            seen.add(value)
            registers[dest] = node

        return cls(values, recipes, roots, users)


class RegisterCompiler:
    '''Try the original order and optionally a dependency-driven schedule.'''

    def prepare(
        self, recipes: dict[int, Operation], roots: set[int],
    ) -> tuple[dict[int, Operation], Counter]:
        required = {}
        remaining = Counter()
        pending = list(roots)
        while pending:
            node = pending.pop()
            if node < 0 or node in required:
                continue
            operation = recipes[node]
            required[node] = operation
            for parent in {operation.left, operation.right}:
                remaining[parent] += 1
                pending.append(parent)
        return required, remaining

    def compile(
        self, puzzle: Puzzle, required: dict[int, Operation], remaining: Counter,
        *, reorder: bool,
    ) -> Program | None:
        remaining = remaining.copy()
        slots = {
            -index - 1: index for index in range(len(puzzle.starts))
            if remaining[-index - 1]
        }
        free = set(range(puzzle.register_count)).difference(slots.values())
        program = []
        fixed = iter(sorted(required))

        if reorder:
            children = {node: [] for node in required}
            parents_left = {}
            for node, operation in required.items():
                computed_parents = {
                    p for p in (operation.left, operation.right) if p >= 0
                }
                parents_left[node] = len(computed_parents)
                for parent in computed_parents:
                    children[parent].append(node)
            ready = {node for node in required if not parents_left[node]}

        def priority(node: int) -> tuple[int, int]:
            operation = required[node]
            released = sum(
                remaining[parent] == 1 for parent in {operation.left, operation.right}
            )
            return int(remaining[node] > 0) - released, node

        for _ in required:
            if reorder:
                node = min(ready, key=priority)
                ready.remove(node)
            else:
                node = next(fixed)
            operation = required[node]
            left, right = slots[operation.left], slots[operation.right]
            for parent in {operation.left, operation.right}:
                remaining[parent] -= 1
                if not remaining[parent]:
                    free.add(slots.pop(parent))
            if not free:
                return None
            dest = free.pop()
            program.append((operation.operator, left, right, dest))
            if remaining[node]:
                slots[node] = dest
            else:
                free.add(dest)
            if reorder:
                for child in children[node]:
                    parents_left[child] -= 1
                    if not parents_left[child]:
                        ready.add(child)
        return tuple(program)


class SharedHelperSearch:
    def __init__(self, calculator: Calculator, *, reorder: bool):
        self.calculator = calculator
        self.compiler = RegisterCompiler()
        self.reorder = reorder
        self.stats = Counter()

    @staticmethod
    def apply(operator: str, left: int, right: int) -> int:
        if operator == '+':
            return left + right
        if operator == '-':
            return left - right
        if operator == '*':
            return left * right
        return left // right

    @staticmethod
    def inverse(result: int, other: int, limit: int):
        if result > other:
            yield result - other, '+', True
        if result + other <= limit:
            yield result + other, '-', True
        if other > result:
            yield other - result, '-', False
        if result % other == 0:
            yield result // other, '*', True
        if result * other <= limit:
            yield result * other, '/', True
        if other % result == 0:
            yield other // result, '/', False

    def routes(
        self, graph: ProgramGraph, output: int, old_helper: int, limit: int,
    ) -> dict[int, list[tuple[str, int | None, bool]]]:
        goal = graph.values[output]
        found = {}
        for node, value in graph.values.items():
            if node >= output or node == old_helper:
                continue
            for helper, operator, helper_first in self.inverse(goal, value, limit):
                found.setdefault(helper, []).append((operator, node, helper_first))
        if goal % 2 == 0:
            found.setdefault(goal // 2, []).append(('+', None, True))
        square = isqrt(goal)
        if square * square == goal:
            found.setdefault(square, []).append(('*', None, True))
        return found

    def build(
        self, target: int, available: dict[int, int], limit: int,
    ) -> Operation | None:
        for value, partner in available.items():
            for operand, operator, operand_first in self.inverse(target, value, limit):
                node = available.get(operand)
                if node is not None:
                    left, right = (node, partner) if operand_first else (partner, node)
                    return Operation(operator, left, right)
        return None

    @staticmethod
    def route(helper: int, spec: tuple[str, int | None, bool]) -> Operation:
        operator, other, helper_first = spec
        if other is None:
            other = helper
        left, right = (helper, other) if helper_first else (other, helper)
        return Operation(operator, left, right)

    @staticmethod
    def forward(helper: int, other: int):
        yield '+', helper + other, True
        yield '*', helper * other, True
        if helper != other:
            yield '-', abs(helper - other), helper > other
        if helper % other == 0:
            yield '/', helper // other, True
        if other % helper == 0:
            yield '/', other // helper, False

    def extended_paths(
        self, graph: ProgramGraph, old: int, direct: int, extended: int,
        direct_routes: dict, extended_routes: dict, available: dict[int, int], limit: int,
    ):
        '''
        Search an extra operation in one branch. Use an existing anchor so the
        longer arm can still pay for itself by removing an old dependency.
        '''
        anchors = sorted(direct_routes.keys() & available.keys())[:HELPER_VALUE_LIMIT]
        for value in anchors:
            helper = available[value]
            emitted = 0
            for partner, partner_value in graph.values.items():
                if partner >= extended or partner == old:
                    continue
                for operator, middle, helper_first in self.forward(value, partner_value):
                    if middle not in extended_routes or not 1 <= middle <= limit:
                        continue
                    middle_node = extended - 1
                    for final in extended_routes[middle][:ROUTES_PER_VALUE]:
                        branch = {
                            middle_node: self.route(helper, (operator, partner, helper_first)),
                            extended: self.route(middle_node, final),
                        }
                        for direct_spec in direct_routes[value][:ROUTES_PER_VALUE]:
                            yield branch | {direct: self.route(helper, direct_spec)}
                            emitted += 1
                        if emitted >= EXTENDED_ROUTES_PER_ANCHOR:
                            break
                    if emitted >= EXTENDED_ROUTES_PER_ANCHOR:
                        break
                if emitted >= EXTENDED_ROUTES_PER_ANCHOR:
                    break

    def candidates(self, puzzle: Puzzle, graph: ProgramGraph, old: int):
        first, second = sorted(graph.users[old])
        limit = puzzle.value_limit
        first_routes = self.routes(graph, first, old, limit)
        second_routes = self.routes(graph, second, old, limit)
        available = {
            value: node for node, value in graph.values.items()
            if node < first and node != old
        }
        options = first_routes.keys() & second_routes.keys()
        ordered = sorted(options, key=lambda value: (value not in available, value))
        for value in ordered[:HELPER_VALUE_LIMIT]:
            helper = available.get(value)
            added = {}
            if helper is None:
                operation = self.build(value, available, limit)
                if operation is None:
                    continue
                helper = first - 1
                added[helper] = operation
            for first_spec in first_routes[value][:ROUTES_PER_VALUE]:
                for second_spec in second_routes[value][:ROUTES_PER_VALUE]:
                    yield added | {
                        first: self.route(helper, first_spec),
                        second: self.route(helper, second_spec),
                    }

        for direct, extended, direct_routes, extended_routes in (
            (first, second, first_routes, second_routes),
            (second, first, second_routes, first_routes),
        ):
            yield from self.extended_paths(
                graph, old, direct, extended, direct_routes, extended_routes, available, limit,
            )

    @staticmethod
    def helpers(graph: ProgramGraph) -> list[int]:
        return [
            node for node, users in graph.users.items()
            if node >= 0 and len(users) == 2 and node not in graph.roots
        ]

    def optimize(self, puzzle: Puzzle, program: Program, graph: ProgramGraph) -> Program:
        best = program
        best_fixed = len(program)
        helpers = sorted(
            self.helpers(graph),
            key=lambda node: (
                -sum(len(graph.users[parent]) == 1
                     for parent in {graph.recipes[node].left, graph.recipes[node].right}),
                node,
            ),
        )
        self.stats['helpers'] += min(len(helpers), HELPER_LIMIT)
        for old in helpers[:HELPER_LIMIT]:
            for changes in self.candidates(puzzle, graph, old):
                self.stats['candidates'] += 1
                '''
                Removing everything that no target needs also drops abandoned
                ancestors of the old helper and its consumers.
                '''
                required, remaining = self.compiler.prepare(graph.recipes | changes, graph.roots)
                if len(required) >= best_fixed:
                    if not self.reorder or len(required) >= len(best):
                        continue
                self.stats['shorter_graphs'] += 1
                candidate = self.compiler.compile(puzzle, required, remaining, reorder=False)
                if candidate is not None:
                    best_fixed = min(best_fixed, len(candidate))
                else:
                    self.stats['fixed_order_allocation_failures'] += 1
                    if self.reorder and len(required) < len(best):
                        candidate = self.compiler.compile(puzzle, required, remaining, reorder=True)
                        if candidate is not None:
                            self.stats['rescheduled_candidates'] += 1
                if candidate is not None and len(candidate) < len(best):
                    self.calculator.replay(puzzle, candidate)
                    best = candidate
        self.stats['fixed_order_savings'] += len(program) - best_fixed
        self.stats['total_savings'] += len(program) - len(best)
        return best


## 4 · Pilot and submission

In [ ]:
class Experiment:
    def __init__(self, config: Config):
        self.portfolio = PortfolioPipeline(config)
        self.search = SharedHelperSearch(self.portfolio.calculator, reorder=ALLOW_REORDER)

    def run(self) -> None:
        start = perf_counter()
        self.portfolio.config.check_inputs()
        puzzles = self.portfolio.load_puzzles()
        programs, profiles = self.portfolio.verify(self.portfolio.config.portfolio_csv, puzzles)
        baseline = sum(profile.length for profile in profiles.values())
        if baseline != EXPECTED_SCORE:
            raise ValueError(f'Unexpected portfolio score: {baseline:,}')

        graphs = {
            pid: ProgramGraph.from_program(puzzles[pid], program)
            for pid, program in programs.items()
        }
        ranked = sorted(
            programs,
            key=lambda pid: (-len(self.search.helpers(graphs[pid])), -len(programs[pid]), pid),
        )
        if PUZZLE_LIMIT is None or PUZZLE_LIMIT >= len(ranked):
            selected = ranked
        else:
            priority_count = (PUZZLE_LIMIT + 1) // 2
            priority = ranked[:priority_count]
            controls = sorted(
                ranked[priority_count:],
                key=lambda pid: (
                    -len(programs[pid]), len(self.search.helpers(graphs[pid])), pid,
                ),
            )[:PUZZLE_LIMIT - priority_count]
            selected = priority + controls

        output = programs.copy()
        wins = []
        for pid in selected:
            candidate = self.search.optimize(puzzles[pid], programs[pid], graphs[pid])
            if len(candidate) < len(programs[pid]):
                output[pid] = candidate
                wins.append((pid, len(programs[pid]) - len(candidate)))
        if wins:
            self.portfolio.export(output, puzzles)

        stats = self.search.stats
        print(f'Puzzles searched: {len(selected):,} | Helpers searched: {stats["helpers"]:,}')
        print(f'Candidate rewrites: {stats["candidates"]:,} | Shorter graphs: {stats["shorter_graphs"]:,}')
        print(f'Fixed-order register failures: {stats["fixed_order_allocation_failures"]:,}')
        print(f'Rescheduled candidates: {stats["rescheduled_candidates"]:,}')
        print(f'Fixed-order savings: {stats["fixed_order_savings"]:,}')
        print(f'Additional scheduling savings: {stats["total_savings"] - stats["fixed_order_savings"]:,}')
        print(f'Baseline: {baseline:,} | Local score: {baseline - stats["total_savings"]:,}')
        print(f'Improved puzzles: {len(wins):,} | Validated rows: {len(output):,}')
        for pid, saved in sorted(wins, key=lambda win: (-win[1], win[0]))[:15]:
            print(f'{pid}: saved {saved}')
        if wins:
            print(f'CSV: {self.portfolio.config.output_csv}')
        else:
            print('No improvement. This run exported no submission CSV.')
        print(f'Elapsed: {perf_counter() - start:.2f}s')

In [ ]:
calculator = Calculator()

pair_puzzle = Puzzle((2, 3, 4), frozenset({8, 10}), 12, 48)
pair_program = (('+', 0, 1, 3), ('+', 3, 1, 4), ('+', 3, 3, 5))
calculator.replay(pair_puzzle, pair_program)
pair_graph = ProgramGraph.from_program(pair_puzzle, pair_program)
search = SharedHelperSearch(calculator, reorder=False)
assert len(search.optimize(pair_puzzle, pair_program, pair_graph)) == 2

In [ ]:
Experiment(Config.discover()).run()